# MotherDuck Silver Layer Validation
Let's verify that the new `dbt-core` transformations actually work in the cloud MotherDuck database!

In [1]:
import duckdb
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()
token = os.getenv('MOTHERDUCK_TOKEN')
db = os.getenv('MD_DATABASE', 'my_db')

con = duckdb.connect(f'md:{db}?token={token}')
print('Connected to MotherDuck!')

Connected to MotherDuck!


### 1. Bronze Layer Counts

In [2]:
con.execute("""
    SELECT 'rent_bronze' as table_name, count(*) as cnt FROM bronze.rent_bronze
    UNION ALL
    SELECT 'sale_bronze', count(*) FROM bronze.sale_bronze
""").df()

,table_name,cnt
0,rent_bronze,14655
1,sale_bronze,16002


### 2. Silver Identity (Unique Listings)

In [3]:
con.execute("""
    SELECT 
        mode,
        count(*) as total_listings,
        min(first_seen_at) as earliest_listing,
        max(last_seen_at) as latest_listing
    FROM silver.listing_identity
    GROUP BY mode
""").df()

,mode,total_listings,earliest_listing,latest_listing
0,sale,1618,2026-04-04 19:13:46.167800+02:00,2026-07-18 07:41:33.566005+02:00
1,rent,2190,2026-04-04 19:13:39.991741+02:00,2026-07-18 07:41:29.991680+02:00


### 3. Silver Versions (SCD Type 2)

In [4]:
con.execute("""
    SELECT 
        mode,
        count(*) as total_versions,
        count(distinct source_listing_id) as unique_listings,
        count(*) - count(distinct source_listing_id) as price_changes_recorded
    FROM silver.listing_versions
    GROUP BY mode
""").df()

,mode,total_versions,unique_listings,price_changes_recorded
0,rent,2550,2190,360
1,sale,1817,1618,199


### 4. Silver Current (Active State)

In [5]:
con.execute("""
    SELECT 
        title, 
        price_total, 
        city, 
        dbt_valid_from
    FROM silver.listing_current 
    ORDER BY dbt_valid_from DESC 
    LIMIT 5
""").df()

,title,price_total,city,dbt_valid_from
0,"54,5 m² | Duży salon | Oddzielna kuchnia | Balkon",667108.0,Kraków,2026-07-18 07:41:33.566005+02:00
1,NIŻSZA CENA - 2 pokoje - 52 m2 - ul. Kazimierz...,799000.0,Kraków,2026-07-18 07:41:33.566005+02:00
2,Dom wolnostojący w doskonałej lokalizacji,1590000.0,Kraków,2026-07-18 07:41:33.566005+02:00
3,Mieszkanie 2 pokojowe ul Bochenka,599900.0,Kraków,2026-07-18 07:41:33.566005+02:00
4,Mieszkanie 47 m² z klimatyzacją | Piasta Tower...,795000.0,Kraków,2026-07-18 07:41:33.566005+02:00


In [6]:
con.close()